# Limpieza y Transformación de Datos
## Proyecto de final de curso — Videojuego (seed42)

**El dominio:** partidas y jugadores de un videojuego multijugador. Las preguntas que quiero responder con los datos: ¿qué mapas y modos dan más puntuación?, ¿qué perfil de jugador tiene más éxito?, ¿cuándo se juega más?, ¿los rangos altos ganan más partidas?

**Este notebook documenta el Paso 2** (limpieza y transformación con Pandas/NumPy) paso a paso: qué problema tenía cada columna en bruto, qué función lo soluciona, y cómo queda el dato antes/después. Todas las funciones viven en `lib/limpieza_datos.py` — aquí solo las llamamos y mostramos el resultado, para que quede documentado el razonamiento.

In [1]:
import pandas as pd
import numpy as np

from lib.cargar_datos import cargar_datos
from lib.limpieza_datos import (
    normalizar_resultado, limpiar_hora, normalizar_personajes_usados,
    limpiar_columna_fecha, limpiar_puntos, detectar_atipicos_iqr,
    corregir_puntos_anomalos, eliminar_duplicados, tratar_nulos_partidas,
    generar_columnas_derivadas_partidas, redondear_sin_decimales,
    normalizar_nick, tratar_nulos_jugadores, generar_columnas_derivadas_jugadores,
    estadisticas_descriptivas,
)

---
## Punto de partida (Paso 1): los datos en bruto

Cargamos los dos CSV originales, tal cual vienen, sin transformar nada todavía.

In [2]:
df_partidas, df_jugadores = cargar_datos(
    "data/videojuego_seed42.csv", "data/videojuego_seed42_jugadores.csv"
)
print("partidas:", df_partidas.shape, "| jugadores:", df_jugadores.shape)
df_partidas.head()

partidas: (10075, 13) | jugadores: (200, 6)


,id_partida,fecha,hora,id_jugador,mapa,modo,personajes_usados,duracion_min,kills,muertes,asistencias,resultado,puntos
0,4181,2026-04-14,19:19,J0109,Bosque Umbral,Casual,Vex,18,4.0,4,6,Victoria,"1120,0 pts"
1,319,24-01-26,13:19,J0069,Estacion Orbital,Casual,Brisa,16,3.0,5,6,Derrota,500.0
2,1316,17/02/2026,19:12,J0073,Bosque Umbral,Ranked,Brisa,30,2.0,5,4,Victoria,800.0
3,1157,2026-02-13,01:03,J0022,Canales de Hielo,Casual,Tesla,13,5.0,2,6,Victoria,1260.0
4,34,2026-01-17,16:13,J0049,Canales de Hielo,Ranked,magnus;tesla;vex,43,4.0,8,1,Derrota,290.0


In [3]:
df_partidas.dtypes

id_partida             int64
fecha                    str
hora                     str
id_jugador               str
mapa                     str
modo                     str
personajes_usados        str
duracion_min           int64
kills                float64
muertes                int64
asistencias            int64
resultado                str
puntos                   str
dtype: object

---
## Explorar antes de tocar nada

Antes de limpiar, miramos qué problemas tiene realmente cada columna — es la única forma de justificar por qué se aplica cada transformación.

In [4]:
print("Valores de 'resultado' (mayúsculas y espacios inconsistentes):")
print(df_partidas['resultado'].unique())
print()
print("Longitudes de 'hora' (algunas traen espacios de más):")
print(df_partidas['hora'].str.len().value_counts())
print()
print("Formatos de 'fecha' mezclados:")
print(df_partidas['fecha'].unique()[:6])
print()
print("'puntos' como texto, con coma decimal y sufijo 'pts':")
print(df_partidas['puntos'].astype(str).sample(5, random_state=1).tolist())
print()
print("Duplicados por id_partida:", df_partidas.duplicated(subset=['id_partida']).sum())
print()
print("Nulos por columna:")
print(df_partidas.isna().sum())

Valores de 'resultado' (mayúsculas y espacios inconsistentes):
<StringArray>
[   'Victoria',     'Derrota',  '  Derrota ',           nan,    'victoria',
    'VICTORIA',     'derrota',     'DERROTA', '  Victoria ']
Length: 9, dtype: str

Longitudes de 'hora' (algunas traen espacios de más):
hora
5    9915
8     160
Name: count, dtype: int64

Formatos de 'fecha' mezclados:
<StringArray>
['2026-04-14',   '24-01-26', '17/02/2026', '2026-02-13', '2026-01-17',
 '21/06/2026']
Length: 6, dtype: str

'puntos' como texto, con coma decimal y sufijo 'pts':
['360.0', '1280.0', '1410.0', '970.0', '1060.0']

Duplicados por id_partida: 75

Nulos por columna:
id_partida             0
fecha                  0
hora                   0
id_jugador             0
mapa                   0
modo                   0
personajes_usados    297
duracion_min           0
kills                300
muertes                0
asistencias            0
resultado            329
puntos               295
dtype: int64


---
## Paso 2.1 — Normalizar texto: `resultado`

`resultado` trae 'Victoria'/'Derrota' mezclado con mayúsculas (`VICTORIA`) y espacios sobrantes (`  Derrota `). Se recorta y se pasa a Title Case. Los nulos se dejan como están a propósito: significan que la partida no registró un desenlace, no hay que inventarlo.

In [5]:
print("Antes:", sorted(df_partidas['resultado'].dropna().unique()))

df = normalizar_resultado(df_partidas)

print("Después:", sorted(df['resultado'].dropna().unique()))

Antes: ['  Derrota ', '  Victoria ', 'DERROTA', 'Derrota', 'VICTORIA', 'Victoria', 'derrota', 'victoria']
Después: ['Derrota', 'Victoria']


---
## Paso 2.2 — Espacios sobrantes en `hora`

160 filas traen la hora con espacios (`'  07:52 '`, 8 caracteres en vez de 5). Sin arreglarlo, MySQL rechaza la carga por superar el ancho de columna.

In [6]:
print("Antes:", df['hora'].str.len().value_counts().to_dict())

df = limpiar_hora(df)

print("Después:", df['hora'].str.len().value_counts().to_dict())

Antes: {5: 9915, 8: 160}
Después: {np.int64(5): 10075}


---
## Paso 2.3 — Campo multivalor: `personajes_usados`

Los personajes de una partida vienen separados por `;`, con mayúsculas inconsistentes por personaje (`magnus;TESLA`). Se normaliza cada uno por separado y se reconstruye la lista. Los nulos pasan a cadena vacía (significa que no se registró ningún personaje, no que falte el dato).

In [7]:
ejemplo_antes = df_partidas.loc[df_partidas['personajes_usados'].str.contains(';', na=False), 'personajes_usados'].iloc[0]
print("Antes:", repr(ejemplo_antes))

df = normalizar_personajes_usados(df)

print("Después:", repr(df.loc[df_partidas['personajes_usados'] == ejemplo_antes, 'personajes_usados'].iloc[0]))

Antes: 'magnus;tesla;vex'
Después: 'Magnus;Tesla;Vex'


---
## Paso 2.4 — Formatos de fecha mixtos

`fecha` mezcla 3 formatos: `AAAA-MM-DD`, `DD/MM/AAAA` y `DD-MM-AA` (año a 2 dígitos). Se prueba cada formato sobre toda la columna a la vez (vectorizado, sin bucles) y se combinan los resultados: cada fecha solo encaja con uno de los tres.

In [8]:
print("Antes (dtype):", df['fecha'].dtype, "| muestras:", df['fecha'].unique()[:4])

df = limpiar_columna_fecha(df)

print("Después (dtype):", df['fecha'].dtype, "| muestras:", df['fecha'].head(4).tolist())

Antes (dtype): str | muestras: <StringArray>
['2026-04-14', '24-01-26', '17/02/2026', '2026-02-13']
Length: 4, dtype: str
Después (dtype): datetime64[us] | muestras: [Timestamp('2026-04-14 00:00:00'), Timestamp('2026-01-24 00:00:00'), Timestamp('2026-02-17 00:00:00'), Timestamp('2026-02-13 00:00:00')]


---
## Paso 2.5 — `puntos`: de texto a número

Algunos valores llevan coma decimal y el sufijo `' pts'` (p.ej. `'1120,0 pts'`). Se limpia el sufijo, se sustituye la coma por punto, y se convierte a numérico.

In [9]:
print("Antes (dtype):", df_partidas['puntos'].dtype)
print("Ejemplos con coma/pts:", df_partidas[df_partidas['puntos'].astype(str).str.contains(',|pts', na=False)]['puntos'].head(3).tolist())

df = limpiar_puntos(df)

print("Después (dtype):", df['puntos'].dtype)

Antes (dtype): str
Ejemplos con coma/pts: ['1120,0 pts', '500,0 pts', '470,0 pts']
Después (dtype): Float64


---
## Paso 2.6 — Valores atípicos en `puntos`

Con el método de Tukey (rango intercuartílico) detectamos valores anormalmente altos (p.ej. `134000` en vez de `1340`) — típico de un '00' añadido por error al capturar el dato. **Regla aplicada:** si el valor es atípico y, al dividirlo entre 100, cae en un rango plausible (0-2000 puntos), se corrige dividiendo entre 100. Si no encaja en ese patrón, se deja como `NaN` para no inventar un dato.

In [10]:
print("describe() ANTES de corregir:")
print(df['puntos'].describe())

df = corregir_puntos_anomalos(df)

print("\ndescribe() DESPUÉS de corregir:")
print(df['puntos'].describe())

describe() ANTES de corregir:
count         9780.0
mean      1099.44274
std      3550.700016
min              0.0
25%            660.0
50%            960.0
75%           1250.0
max         134000.0
Name: puntos, dtype: Float64
[Paso 2] 'puntos': 60 corregidos (÷100), 0 pasados a NaN

describe() DESPUÉS de corregir:
count        9780.0
mean     957.075072
std      410.822474
min             0.0
25%           660.0
50%           950.0
75%          1250.0
max          2130.0
Name: puntos, dtype: Float64


---
## Paso 2.7 — Duplicados

Filas totalmente repetidas por `id_partida` (probablemente un error al exportar el CSV original). Se conserva la primera aparición de cada una.

In [11]:
print("Filas antes:", len(df), "| duplicados detectados:", df.duplicated(subset=['id_partida']).sum())

df = eliminar_duplicados(df, subset=["id_partida"])

print("Filas después:", len(df))

Filas antes: 10075 | duplicados detectados: 75
[Paso 2] Duplicados eliminados: 75
Filas después: 10000


---
## Paso 2.8 — Valores nulos

`kills` tiene nulos que se imputan con la **mediana** (poco sesgo frente a outliers). `resultado` y `puntos`, en cambio, **no** se tocan aquí: sus nulos tienen significado propio (partida sin desenlace registrado / sin puntuación registrada) y rellenarlos sería inventar un dato que no existe.

In [12]:
print("Nulos en 'kills' antes:", df['kills'].isna().sum())

df = tratar_nulos_partidas(df)

print("Nulos en 'kills' después:", df['kills'].isna().sum())
print("Nulos en 'resultado' (se dejan, es intencional):", df['resultado'].isna().sum())

Nulos en 'kills' antes: 300
Nulos en 'kills' después: 0
Nulos en 'resultado' (se dejan, es intencional): 329


---
## Paso 2.9 — Columnas derivadas

A partir de las columnas ya limpias, se generan nuevas columnas útiles para el análisis: `anio`, `mes`, `dia_semana` (de `fecha`), `kd_ratio` (kills/muertes, protegido contra división por 0) y `num_personajes` (cuántos personajes se usaron en la partida).

In [13]:
df = generar_columnas_derivadas_partidas(df)

columnas_nuevas = [c for c in df.columns if c not in df_partidas.columns]
print("Columnas nuevas:", columnas_nuevas)
df[columnas_nuevas].head()

Columnas nuevas: ['anio', 'mes', 'dia_semana', 'kd_ratio', 'num_personajes']


,anio,mes,dia_semana,kd_ratio,num_personajes
0,2026,4,Tuesday,1.0,1
1,2026,1,Saturday,0.6,1
2,2026,2,Tuesday,0.4,1
3,2026,2,Friday,2.5,1
4,2026,1,Saturday,0.5,3


---
## Paso 2.10 — Redondeo sin decimales

`kills`, `puntos` y `kd_ratio` se redondean a 0 decimales y se convierten a `Int64` (el tipo entero de pandas que admite nulos), para que el CSV final no guarde `957.1428571`.

In [14]:
df = redondear_sin_decimales(df, ["kills", "puntos", "kd_ratio"])

df[["kills", "puntos", "kd_ratio"]].head()

,kills,puntos,kd_ratio
0,4,1120,1
1,3,500,1
2,2,800,0
3,5,1260,2
4,4,290,0


**`df` ahora es exactamente el resultado de `limpiar_partidas()`** — hemos hecho a mano, paso a paso, lo que esa función hace de una vez en `main.py`.

---
## Ahora la tabla `jugadores`

Menos pasos, pero uno de ellos (el `nick`) es el más elaborado de todo el proyecto.

In [15]:
df_jugadores.head()

,id_jugador,nick,region,nivel,rango,fecha_registro
0,J0001,amayamanuela,EU-Oeste,95,Plata,2024-03-23
1,J0002,jose-ignacio32,EU-Oeste,14,Oro,2025-05-23
2,J0003,bonetdominga,EU-Este,76,Plata,2023-07-08
3,J0004,banton,EU-Oeste,30,Plata,2025-12-07
4,J0005,giljoan,NaN,92,Oro,2025-12-10


### Paso 2.11 — `nick`: nombre y apellido pegados sin espacio

El `nick` no es un nombre roto al azar: es un nombre y un apellido concatenados sin espacio (p.ej. `'amayamanuela'`), a veces en orden nombre+apellido, a veces apellido+nombre, a veces solo la inicial de uno de los dos — típico de un generador automático de usuarios. `normalizar_nick()` usa un diccionario de nombres/apellidos españoles (construido a partir de los propios datos) para separarlos y devolver siempre "Nombre Apellido" completo, sin dejar iniciales sueltas.

In [16]:
print("Antes:", df_jugadores['nick'].head(5).tolist())

dj = normalizar_nick(df_jugadores.copy())

print("Después:", dj['nick'].head(5).tolist())

Antes: ['amayamanuela', 'jose-ignacio32', 'bonetdominga', 'banton', 'giljoan']
Después: ['Manuela Amaya', 'Jose Ignacio Dalmau', 'Dominga Bonet', 'Anton Carrion', 'Joan Gil']


### Paso 2.12 — Duplicados y región nula

Igual que en partidas, se eliminan duplicados por `id_jugador`. La región nula se rellena con una región elegida al azar **entre las que ya existen** en el propio dataset (respetando sus proporciones reales), en vez de inventar una categoría "Desconocida" que no aporta información real para agrupar.

In [17]:
print("Nulos en 'region' antes:", dj['region'].isna().sum())

dj = eliminar_duplicados(dj, subset=["id_jugador"])
dj = tratar_nulos_jugadores(dj)

print("Nulos en 'region' después:", dj['region'].isna().sum())

Nulos en 'region' antes: 40
[Paso 2] Duplicados eliminados: 0
Nulos en 'region' después: 0


### Paso 2.13 — Columna derivada: antigüedad

Se calcula cuántos días lleva cada jugador registrado, a partir de `fecha_registro`.

In [18]:
dj = generar_columnas_derivadas_jugadores(dj)

dj[["fecha_registro", "antiguedad_dias"]].head()

,fecha_registro,antiguedad_dias
0,2024-03-23,857
1,2025-05-23,431
2,2023-07-08,1116
3,2025-12-07,233
4,2025-12-10,230


---
## Estadísticas descriptivas con NumPy

Media, mediana, desviación típica, mínimo y máximo de las columnas numéricas — calculadas con `np.nanmean`, `np.nanmedian`, `np.nanstd`, `np.nanmin`, `np.nanmax` sobre toda la matriz a la vez (sin bucles), ignorando nulos automáticamente.

In [19]:
columnas_numericas = ["duracion_min", "kills", "muertes", "asistencias", "puntos", "kd_ratio"]
estadisticas_descriptivas(df, columnas_numericas)

,media,mediana,desviacion_tipica,minimo,maximo
duracion_min,26.124600,26.0,9.736410,8.0,45.0
kills,6.028800,6.0,3.035716,0.0,20.0
muertes,4.990300,5.0,2.232130,0.0,19.0
asistencias,4.006400,4.0,1.988104,0.0,14.0
puntos,957.135174,950.0,410.695049,0.0,2130.0
kd_ratio,1.558002,1.0,1.437742,0.0,16.0


---
## Guardar los datos limpios

Esta es la versión que se carga en MySQL en el Paso 3.

In [20]:
df.to_csv("data/partidas_limpio.csv", index=False)
dj.to_csv("data/jugadores_limpio.csv", index=False)

print("Guardado: data/partidas_limpio.csv y data/jugadores_limpio.csv")

Guardado: data/partidas_limpio.csv y data/jugadores_limpio.csv


---
## Resumen de decisiones tomadas

| Problema | Decisión | Por qué |
|---|---|---|
| Mayúsculas/espacios inconsistentes | Normalizar a Title Case | Para poder agrupar/filtrar sin duplicar categorías |
| 3 formatos de fecha mezclados | Probar cada formato y combinar | Vectorizado, sin bucles, cubre el 100% de los casos |
| `puntos` como texto con coma y 'pts' | Limpiar sufijo y coma, convertir a numérico | Necesario para poder calcular medias/sumas |
| Valores atípicos en `puntos` (÷100) | Corregir si el patrón encaja, si no `NaN` | No se inventa un dato que no se puede reconstruir con certeza |
| Duplicados exactos | Eliminar, quedarse con el primero | Error de exportación, no información real |
| Nulos en `kills` | Imputar con la mediana | Poco sesgo frente a outliers |
| Nulos en `resultado` / `puntos` | Dejar como `NaN` | Significan "no registrado", no "cero" |
| Nulos en `region` | Relleno aleatorio proporcional | Mejor que una categoría "Desconocida" sin valor analítico |
| `nick` pegado sin espacio | Diccionario nombre/apellido + reglas | Es información recuperable, no hay que descartarla |
